# Analyse Économique de la Lutte Sénégalaise (Lamb)

# Partie 3 — Transformations avancées + Optimisation + Machine Learning

## Objectif

Cette partie exploite le dataset nettoyé `lamb_clean.csv` produit en Partie 1.

Nous allons appliquer :

- jointures avancées
- agrégations complexes
- window functions
- optimisation Spark
- Machine Learning **exploratoire**

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("Partie3_Lamb")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions","8")
         .getOrCreate())

# Étape 1 — Chargement du dataset propre

Nous utilisons directement `lamb_clean.csv`, issu du pipeline Data Engineering.

In [ ]:
df = spark.read.option("header",True).option("inferSchema",True).csv("/content/lamb_clean.csv")

print("Nombre de lignes :", df.count())
print("Nombre de colonnes :", len(df.columns))

df.show(5)

Nombre de lignes : 2000
Nombre de colonnes : 15
+--------+----------+----+-----+----------+----------+------+------------+----------------+-----------------+------------+-----------+---------------+-------------+------------------+
|fight_id|      date|year|month|wrestler_1|wrestler_2|winner|winner_label|           arena|         promoter|tickets_sold|    revenue|sponsor_revenue|total_revenue|has_ticket_revenue|
+--------+----------+----+-----+----------+----------+------+------------+----------------+-----------------+------------+-----------+---------------+-------------+------------------+
|       0|2024-01-25|2024|    1|         3|         0|  NULL|  Nul/Annulé| Arène nationale|Promoteur inconnu|        8314|1.9456904E7|        1670528|  2.1127432E7|              true|
|       1|2022-08-27|2022|    8|         2|         9|   9.0|           9|  Arène inconnue|Promoteur inconnu|        4070|  3132899.0|        2334068|    5466967.0|              true|
|       2|2022-08-14|2022|    8|

# Étape 2 — Jointure avec table des promoteurs

Création d'une table de référence pour enrichir les promoteurs.

In [ ]:
promoters = spark.createDataFrame([
("Gaston Mbengue","Historique"),
("Promoteur inconnu","Non classé")
],["promoter","categorie_promoter"])

df_join = df.join(promoters,on="promoter",how="left")

df_join.select("promoter","categorie_promoter").show()

+-----------------+------------------+
|         promoter|categorie_promoter|
+-----------------+------------------+
|      Aziz Ndiaye|              NULL|
|      Aziz Ndiaye|              NULL|
|      Aziz Ndiaye|              NULL|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|   Gaston Mbengue|        Historique|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
|Promoteur inconnu|        Non classé|
+-----------------+------------------+
only showing top 20 rows


# Étape 3 — Agrégation avancée

Analyse du chiffre d'affaires par promoteur et par année.

In [ ]:
agg = (df_join
.groupBy("year","promoter")
.agg(
    F.count("*").alias("nb_combats"),
    F.sum("total_revenue").alias("ca_total"),
    F.avg("total_revenue").alias("ca_moyen")
)
.orderBy(F.desc("ca_total"))
)

agg.show(20,False)

+----+-----------------+----------+-------------+--------------------+
|year|promoter         |nb_combats|ca_total     |ca_moyen            |
+----+-----------------+----------+-------------+--------------------+
|2023|Promoteur inconnu|279       |3.257094516E9|1.1674173892473118E7|
|2023|Aziz Ndiaye      |259       |3.158870968E9|1.2196413003861004E7|
|2022|Aziz Ndiaye      |285       |3.151180605E9|1.105677405263158E7 |
|2022|Gaston Mbengue   |277       |3.042312852E9|1.0983078888086643E7|
|2022|Promoteur inconnu|265       |2.859988427E9|1.0792409158490567E7|
|2023|Gaston Mbengue   |259       |2.20819736E9 |8525858.532818533   |
|2024|Gaston Mbengue   |135       |1.779776536E9|1.3183529896296296E7|
|2024|Aziz Ndiaye      |130       |1.278319595E9|9833227.653846154   |
|2024|Promoteur inconnu|111       |1.256096347E9|1.1316183306306306E7|
+----+-----------------+----------+-------------+--------------------+



# Étape 4 — Window Function : classement annuel

Classement des promoteurs selon le chiffre d'affaires annuel.

In [ ]:
windowSpec = Window.partitionBy("year").orderBy(F.desc("ca_total"))

ranking = agg.withColumn(
    "rang_annuel",
    F.dense_rank().over(windowSpec)
)

ranking.show()

+----+-----------------+----------+-------------+--------------------+-----------+
|year|         promoter|nb_combats|     ca_total|            ca_moyen|rang_annuel|
+----+-----------------+----------+-------------+--------------------+-----------+
|2022|      Aziz Ndiaye|       285|3.151180605E9| 1.105677405263158E7|          1|
|2022|   Gaston Mbengue|       277|3.042312852E9|1.0983078888086643E7|          2|
|2022|Promoteur inconnu|       265|2.859988427E9|1.0792409158490567E7|          3|
|2023|Promoteur inconnu|       279|3.257094516E9|1.1674173892473118E7|          1|
|2023|      Aziz Ndiaye|       259|3.158870968E9|1.2196413003861004E7|          2|
|2023|   Gaston Mbengue|       259| 2.20819736E9|   8525858.532818533|          3|
|2024|   Gaston Mbengue|       135|1.779776536E9|1.3183529896296296E7|          1|
|2024|      Aziz Ndiaye|       130|1.278319595E9|   9833227.653846154|          2|
|2024|Promoteur inconnu|       111|1.256096347E9|1.1316183306306306E7|          3|
+---

# Étape 5 — Revenu cumulé mensuel

Calcul du revenu cumulé au fil des mois.

In [ ]:
monthly = (df
.groupBy("year","month")
.agg(F.sum("total_revenue").alias("revenu_mensuel"))
)

windowCum = Window.partitionBy("year").orderBy("month")

monthly = monthly.withColumn(
    "revenu_cumule",
    F.sum("revenu_mensuel").over(windowCum)
)

monthly.show()

+----+-----+--------------+-------------+
|year|month|revenu_mensuel|revenu_cumule|
+----+-----+--------------+-------------+
|2022|    1|  8.65270086E8| 8.65270086E8|
|2022|    2|  4.40948205E8|1.306218291E9|
|2022|    3|  6.75983775E8|1.982202066E9|
|2022|    4|  8.43685724E8| 2.82588779E9|
|2022|    5|  7.16323431E8|3.542211221E9|
|2022|    6|  8.01169895E8|4.343381116E9|
|2022|    7|  8.16570852E8|5.159951968E9|
|2022|    8|  7.46003476E8|5.905955444E9|
|2022|    9|  8.95557815E8|6.801513259E9|
|2022|   10|  7.78703168E8|7.580216427E9|
|2022|   11|  9.30180141E8|8.510396568E9|
|2022|   12|  5.43085316E8|9.053481884E9|
|2023|    1|   6.2151536E8|  6.2151536E8|
|2023|    2|  9.05199859E8|1.526715219E9|
|2023|    3|  4.76925202E8|2.003640421E9|
|2023|    4|  6.77262153E8|2.680902574E9|
|2023|    5|  7.29460226E8|  3.4103628E9|
|2023|    6|  6.63489154E8|4.073851954E9|
|2023|    7|  6.39769943E8|4.713621897E9|
|2023|    8|  6.18928714E8|5.332550611E9|
+----+-----+--------------+-------

# Étape 6 — Optimisation Spark

Utilisation du cache et visualisation du plan d'exécution.

In [ ]:
df.cache()

df.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: csv, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
fight_id: int, date: date, year: int, month: int, wrestler_1: int, wrestler_2: int, winner: double, winner_label: string, arena: string, promoter: string, tickets_sold: int, revenue: double, sponsor_revenue: int, total_revenue: double, has_ticket_revenue: boolean
Relation [fight_id#113,date#114,year#115,month#116,wrestler_1#117,wrestler_2#118,winner#119,winner_label#120,arena#121,promoter#122,tickets_sold#123,revenue#124,sponsor_revenue#125,total_revenue#126,has_ticket_revenue#127] csv

== Optimized Logical Plan ==
InMemoryRelation [fight_id#113, date#114, year#115, month#116, wrestler_1#117, wrestler_2#118, winner#119, winner_label#120, arena#121, promoter#122, tickets_sold#123, revenue#124, sponsor_revenue#125, total_revenue#126, has_ticket_revenue#127], StorageLevel(disk, memory, deserialized, 1 replicas)
   +- FileScan csv [fight_id#113,date#114,year#115,mo

# Étape 7 — Machine Learning (Segmentation des combats)

Utilisation de KMeans pour segmenter les combats selon :

- total_revenue
- tickets_sold
- sponsor_revenue

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

features = ["total_revenue","tickets_sold","sponsor_revenue"]

assembler = VectorAssembler(inputCols=features, outputCol="features")

data_ml = assembler.transform(df).select("features")

kmeans = KMeans(k=3, seed=1)
model = kmeans.fit(data_ml)

predictions = model.transform(data_ml)

predictions.show(10,False)

+-------------------------------+----------+
|features                       |prediction|
+-------------------------------+----------+
|[2.1127432E7,8314.0,1670528.0] |2         |
|[5466967.0,4070.0,2334068.0]   |0         |
|[4019187.0,18856.0,4019187.0]  |0         |
|[3354228.0,14848.0,3354228.0]  |0         |
|[2.9881797E7,12763.0,3385309.0]|2         |
|[2.9362369E7,19089.0,2959357.0]|2         |
|[5.0694765E7,2501.0,2411749.0] |1         |
|[3560434.0,15857.0,3560434.0]  |0         |
|[1935615.0,3339.0,1935615.0]   |0         |
|[2342265.0,19250.0,2342265.0]  |0         |
+-------------------------------+----------+
only showing top 10 rows


# Étape 8 — Export final

Sauvegarde des résultats au format Parquet.

In [ ]:
ranking.write.mode("overwrite").parquet("/content/resultat_partie3")

# Conclusion

Cette partie a permis de mobiliser les concepts avancés de PySpark :

- jointures distribuées
- agrégations multi-niveaux
- window functions
- optimisation Catalyst
- Machine Learning distribué

Le dataset Lamb permet ainsi une analyse économique complète du secteur.